# Day 067 — Exercise 5: Analyze Image Pipeline

**What you'll build:** `analyze_image_pipeline(img, tasks, describe_fn=None)` — encode once, run multiple vision analyses, return a results dict.

**Why it matters:** Encoding is CPU-bound — doing it once for multiple analyses is more efficient than re-encoding per task. The dict output serialises cleanly to JSON, logs easily, and is straightforward to pass downstream for structured processing.

In [ ]:
import io
import base64
from PIL import Image

def image_to_base64(img: Image.Image, format: str = 'PNG') -> str:
    buf = io.BytesIO()
    out = img
    if format.upper() in ('JPEG', 'JPG') and img.mode in ('RGBA', 'P'):
        out = img.convert('RGB')
    out.save(buf, format=format)
    return base64.b64encode(buf.getvalue()).decode()

import ollama

_TASKS_PROMPTS = {
    'describe': 'Describe this image in 2-3 sentences.',
    'text':     ('Extract all visible text from this image exactly as it '
                 'appears. If there is no text, reply with an empty string.'),
    'colors':   'List the 3 dominant colors visible in this image.',
    'objects':  'List the main objects visible in this image.',
}

def describe_image(img_b64: str, prompt: str = 'Describe this image.',
                   describe_fn=None) -> str:
    if describe_fn is not None:
        return describe_fn(img_b64, prompt)
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': prompt, 'images': [img_b64]}]
    )
    return resp['message']['content']

_test_img = Image.new('RGB', (150, 150), color=(0, 100, 200))


## Task

Implement `analyze_image_pipeline(img, tasks, describe_fn=None) -> dict`:

1. Encode the image **once**: `img_b64 = image_to_base64(img)`
2. For each task in `tasks`:
   - Raise `ValueError` if the task is not in `_TASKS_PROMPTS`
   - Call `describe_image(img_b64, _TASKS_PROMPTS[task], describe_fn=describe_fn)`
   - Store the result in `results[task]`
3. Return the `results` dict

## Your Implementation

In [ ]:
def analyze_image_pipeline(img, tasks: list,
                            describe_fn=None) -> dict:
    """Run multiple vision analyses on a single image.

    Encodes the image once, then runs each task from _TASKS_PROMPTS.

    Args:
        img:         PIL Image to analyze
        tasks:       list of task names — keys from _TASKS_PROMPTS
        describe_fn: callable(img_b64, prompt) -> str for testing
    Returns:
        dict mapping each task name to its result string
    Raises:
        ValueError for any unknown task name
    """
    raise NotImplementedError


In [ ]:
def analyze_image_pipeline(img, tasks: list,
                            describe_fn=None) -> dict:
    img_b64 = image_to_base64(img)
    results = {}
    for task in tasks:
        if task not in _TASKS_PROMPTS:
            raise ValueError(
                f'Unknown task: {task!r}. Available: {list(_TASKS_PROMPTS)}'
            )
        results[task] = describe_image(
            img_b64, _TASKS_PROMPTS[task], describe_fn=describe_fn
        )
    return results


## Automated checks

In [ ]:
score, total = 0, 5
try:
    _mock = lambda b, p: f'result for: {p[:20]}'

    # Returns dict
    result = analyze_image_pipeline(_test_img, ['describe'], describe_fn=_mock)
    assert isinstance(result, dict), f"Expected dict, got {type(result)}"
    score += 1; print("\u2705 returns a dict")

    # Keys match requested tasks
    assert set(result.keys()) == {'describe'}, (
        f"Expected keys {{'describe'}}, got {set(result.keys())}")
    score += 1; print("\u2705 dict keys match the requested tasks")

    # Multi-task
    multi = analyze_image_pipeline(_test_img, ['describe', 'colors'],
                                    describe_fn=_mock)
    assert set(multi.keys()) == {'describe', 'colors'}
    assert all(isinstance(v, str) for v in multi.values())
    score += 1; print("\u2705 multi-task pipeline returns all requested tasks")

    # Empty tasks list → empty dict
    empty = analyze_image_pipeline(_test_img, [], describe_fn=_mock)
    assert empty == {}, f"Empty tasks should return empty dict, got {empty}"
    score += 1; print("\u2705 empty tasks list returns empty dict")

    # Unknown task raises ValueError
    raised = False
    try:
        analyze_image_pipeline(_test_img, ['describe', 'mood'], describe_fn=_mock)
    except ValueError:
        raised = True
    assert raised, "Unknown task should raise ValueError"
    score += 1; print("\u2705 unknown task raises ValueError")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def analyze_image_pipeline(img, tasks: list,
                            describe_fn=None) -> dict:
    img_b64 = image_to_base64(img)
    results = {}
    for task in tasks:
        if task not in _TASKS_PROMPTS:
            raise ValueError(
                f'Unknown task: {task!r}. Available: {list(_TASKS_PROMPTS)}'
            )
        results[task] = describe_image(
            img_b64, _TASKS_PROMPTS[task], describe_fn=describe_fn
        )
    return results
```

**Why encode once?** PIL save + base64 encode touches CPU for every byte of the image. For a 1 megapixel image (~3 MB uncompressed), encoding twice doubles that work. More importantly, `image_to_base64` called inside the loop would produce the same base64 every time — wasted work. Encode at the pipeline boundary, pass the string inward.

</details>